# GCAP3226 Week 3 — Graded in-class exercise (5%)

**Individual** · submit via **your GitHub fork** (end of notebook). Keep answers short.

### What is marked
| Task | Focus |
|------|--------|
| **1** | Stacked bar: one factor **other than** `government_consideration` vs `support_info` + your trend description |
| **2** | Multiple linear regression that builds on the demo, adding `policy_helpfulness` and `waste_severity` + short coefficient interpretation |
| **3** | **Your own** IPO prompt → simple linear regression: `support_info` vs `fairness` |
| **Submit** | Save → Source Control → **Commit & Push** → **commit message** → Commit → Moodle **GitHub URL** |

Use **association** language only. This is a **non-probability** sample — do not claim causation or “all of Hong Kong”.


## How to use AI here (Codespaces)

In the demo you practiced **Input → Process → Output** prompts (**Write Python code** in Process).

**Tasks 1–2:** the prompt is already in the code cell as comments.

**Task 3:** you **write the prompt yourself** in the comments, then use Inline Chat.

Workflow:
1. Click inside the code cell.  
2. Edit / write the comment prompt.  
3. Press **`Cmd+I`** (Mac) or **`Ctrl+I`** (Windows/Linux).  
4. Ask e.g. *Implement the prompt in the comments above* → **Keep**.  
5. Fix anything that breaks; write interpretations **in your own words**.


## Setup (run once)

Load `week3.csv`. Glance at the column names so you know what you can use.


In [ ]:
# Setup — run this cell first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LinearRegression

DATA_FILE = "week3.csv"


def load_week3_data():
    for p in [Path(DATA_FILE), Path.cwd() / DATA_FILE]:
        if p.is_file():
            print("Using data file:", p.resolve())
            return pd.read_csv(p)
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Work in your Week 3 fork / Codespace."
    )


df = load_week3_data()
print("shape:", df.shape)
print("columns:", list(df.columns))
df.head()


## Task 1 — Stacked bar (not the demo pair)

In the demo, the stacked bar used **`government_consideration`** (groups) × **`support_info`** (stack, row %).

**Your job:** choose a **different** grouping variable from `week3.csv` and plot a **row-% stacked bar** of `support_info` within each group.

### Suggested grouping variables (pick one)
| Variable | Notes |
|----------|--------|
| `fairness` | 1–5 |
| `policy_helpfulness` | 1–4 |
| `waste_severity` | 1–4 |
| `recycling_effort` | 1–4 |
| `education` | 1–4 |

Do **not** use `government_consideration` as the grouping variable.

In the **next code cell**, replace `<YOUR_GROUP_VAR>` in the comment prompt with your choice, then use Inline Chat (see **How to use AI** above).


In [ ]:
# Task 1 — Stacked bar
# --- Sample AI prompt (Input → Process → Output) ---
# Input: dataframe df already loaded from week3.csv.
# Process: Write Python code to make a row-percentage stacked bar chart of
#   support_info within each level of fairness.
#   Drop rows with missing values for these columns. Show n for each group if you can.
#   Add comments to key Python code.
# Output: A stacked bar chart with title, axis labels, and legend.
# --- End prompt ---
# (AI 用法:光标放本 cell → Ctrl+I → "Implement the prompt in the comments above" → Keep)

import numpy as np
import matplotlib.pyplot as plt

# Keep the two columns we need; drop missing rows
sub = df[["fairness", "support_info"]].dropna()

# Frequency table: rows = fairness level, columns = support level (1-5)
tab = pd.crosstab(sub["fairness"], sub["support_info"])

# Row percentages: each fairness group adds up to 100%
pct = tab.div(tab.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(7.5, 4.5))
bottom = np.zeros(len(pct))
for i, lv in enumerate(pct.columns):
    ax.bar(pct.index.astype(str), pct[lv].values, bottom=bottom,
           label="support_info = %s" % lv)
    bottom += pct[lv].values

# Show n inside the x tick labels so thin groups stay visible
ax.set_xticks(range(len(pct.index)))
ax.set_xticklabels(["%s\n(n=%d)" % (g, int(tab.loc[g].sum())) for g in pct.index])

ax.set_xlabel("fairness (1 = not fair ... 5 = very fair)")
ax.set_ylabel("Row % within each fairness group")
ax.set_title("Support for MSW charging by perceived fairness (row %, n=97)")
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

# Numbers you can quote in the write-up
print(tab)
print(pct.round(1))


### Task 1 — Describe the trend (your words)

In 2–4 sentences: what pattern do you see in **this sample**? Use association language. Mention any thin groups (small *n*) if relevant.

**How to answer:** type inside the `""" ... """` in the next cell, then **run** that cell so your text appears in the output.


In [ ]:
task1_trend = """
In this sample, support for MSW charging rises with perceived fairness: among respondents who
rate the scheme least fair (fairness = 1, n = 15), 86.7% are at support_info = 1, while among
those who rate it fair (fairness = 4, n = 35) 62.9% choose 4 or 5. The fairness = 5 group shows
75% at support 5, but it has only 4 respondents, so that percentage is very unstable.
This is a positive association in this sample only - it does not show that fairness causes
support, and the sample is non-probability (n = 97).
"""
print(task1_trend)


## Task 2 — Multiple linear regression (+ policy & severity)

In the demo you fitted multiple linear regression with `government_consideration` and `fairness` (and a fuller model with demographics).

**Your job:** fit a multiple linear regression for this sample:

**Response:** `support_info`  

**Exploratory variables (at least these four):**
- `government_consideration`
- `fairness`
- `policy_helpfulness`
- `waste_severity`

Use complete cases for these columns. Print **each coefficient**, **R²**, and **n**.  

In the **next code cell**, use the comment prompt with Inline Chat (see **How to use AI** above).


In [ ]:
# Task 2 — Multiple linear regression
# --- Sample AI prompt (Input → Process → Output) ---
# Input: dataframe df already loaded from week3.csv.
# Process: Write Python code to run a multiple linear regression with
#   response = support_info and exploratory variables =
#   government_consideration, fairness, policy_helpfulness, waste_severity.
#   Use rows with no missing values on these columns.
#   Print coefficients, R-squared, and n.
#   Add comments to key Python code.
# Output: A clear coefficient table (or printed list), R², and sample size n.
# --- End prompt ---

from sklearn.linear_model import LinearRegression

XS = ["government_consideration", "fairness", "policy_helpfulness", "waste_severity"]
d2 = df[["support_info"] + XS].dropna()          # complete cases only

model = LinearRegression().fit(d2[XS], d2["support_info"])

print("n =", len(d2))
print("R2 =", round(model.score(d2[XS], d2["support_info"]), 3))
print("intercept =", round(model.intercept_, 3))
for name, coef in zip(XS, model.coef_):
    print("  %-24s % .3f" % (name, coef))

# Optional extra: p-values (statsmodels is installed in this Codespace)
try:
    import statsmodels.api as sm
    m = sm.OLS(d2["support_info"], sm.add_constant(d2[XS])).fit()
    print("\np-values:")
    for k in ["const"] + XS:
        print("  %-24s p = %.4f" % (k, m.pvalues[k]))
except Exception as e:
    print("statsmodels not available:", e)


### Task 2 — Report & briefly interpret

1. Paste or list the **coefficients** (and R², n) from your output.  
2. In a few sentences, interpret **two** coefficients that interest you (association; holding the other variables fixed).  
3. One caution (sample / overclaim).

**How to answer:** type inside the `""" ... """` in the next cell, then **run** that cell so your text appears in the output.


In [ ]:
task2_writeup = """
# Coefficients / R2 / n:
# intercept = -0.277 ; government_consideration = 0.483 (p = 0.0001) ;
# fairness = 0.379 (p = 0.0004) ; policy_helpfulness = 0.192 (p = 0.089) ;
# waste_severity = 0.118 (p = 0.404) ; R2 = 0.639 ; n = 97

# Interpretation (2 coefficients I care about):
# Holding the other three variables fixed, each one-point higher rating of
# "government considered diverse opinions" is associated with about 0.48 points higher initial
# support for MSW charging. Holding the others fixed, each one-point higher fairness rating is
# associated with about 0.38 points higher support. The other two variables are positive but small
# and not distinguishable from zero at the 5% level (p = 0.089 and p = 0.404).

# One caution:
# This is a cross-sectional, self-reported, non-probability sample (n = 97) - association only,
# no causation, and no claim about all of Hong Kong. fairness and government_consideration are
# strongly correlated (r = 0.73), so their separate coefficients are less precise, and the 1-5
# scale is ordinal but treated here as a number.
"""
print(task2_writeup)


## Task 3 — Write your own prompt (simple linear regression)

**Goal:** `support_info` (response) vs `fairness` (one exploratory variable) — a **simple** linear regression on this sample.

Unlike Tasks 1–2, **do not** copy a ready-made Process. In the **next code cell**, write the full **Input / Process / Output** in the comments (**Process** must include **Write Python code**), then use Inline Chat (see **How to use AI** above).

Keep the model to **these two variables only** (not the Task 2 multiple linear regression). You should get coefficients, R², and n.


In [ ]:
# Task 3 — YOUR prompt
# --- Write your AI prompt here (Input → Process → Output) ---
# Input: dataframe df already loaded from week3.csv (97 respondents).
# Process: Write Python code to run a SIMPLE linear regression with
#   response = support_info and ONE exploratory variable = fairness.
#   Use rows with no missing values on these two columns.
#   Print the intercept, the fairness coefficient, R-squared and n, then draw a scatter plot
#   of the two variables with the fitted line.
# Output: Printed intercept / slope / R2 / n, and a scatter plot with the regression line.
# --- End prompt ---
# Requirement: simple linear regression only — response = support_info, exploratory = fairness.

from sklearn.linear_model import LinearRegression

d3 = df[["support_info", "fairness"]].dropna()
simple = LinearRegression().fit(d3[["fairness"]], d3["support_info"])

print("n =", len(d3))
print("R2 =", round(simple.score(d3[["fairness"]], d3["support_info"]), 3))
print("intercept =", round(simple.intercept_, 3))
print("fairness slope =", round(simple.coef_[0], 3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(d3["fairness"], d3["support_info"], alpha=0.4)
xs = np.linspace(d3["fairness"].min(), d3["fairness"].max(), 50)
ax.plot(xs, simple.intercept_ + simple.coef_[0] * xs, color="red")
ax.set_xlabel("fairness (1-5)")
ax.set_ylabel("support_info (1-5)")
ax.set_title("support_info vs fairness - simple linear regression (n = 97)")
plt.tight_layout()
plt.show()


### Task 3 — Short note

In 1–2 sentences: what does the **fairness** slope mean in this simple linear regression? How does it compare (roughly) with the fairness coefficient in your **Task 2** multiple linear regression?

**How to answer:** type inside the `""" ... """` in the next cell, then **run** that cell.


In [ ]:
task3_note = """
# Fairness slope (simple linear regression) and vs Task 2:
# Simple: support_info = 0.495 + 0.794 * fairness (R2 = 0.522, n = 97) - each one-point higher
# fairness rating goes with about 0.79 points higher support.
# In the Task 2 multiple regression the fairness coefficient drops to 0.379, because fairness and
# government_consideration move together (r = 0.73): once "government considered diverse opinions"
# is in the model, part of what looked like the fairness effect is already explained by it.
# Both coefficients stay positive, and both are statistically significant in this sample.
"""
print(task3_note)


## How to submit

Submit the **GitHub URL** of this notebook in **your fork** (not an `.ipynb` upload).

1. **Save** the notebook.  
2. Open **Source Control** (left sidebar, branch icon).  
3. Click **Commit & Push**.  
4. Type a **commit message** → click **Commit**.  
5. On github.com, open the file in **your fork** → copy the browser URL.  
6. Paste that URL on **Moodle**.

**If push fails:** ask a TA — keep working toward a push to your fork.

### Checklist
- [ ] Task 1: stacked bar (not `government_consideration` groups) + trend description  
- [ ] Task 2: multiple linear regression with the four required variables + coefficient write-up  
- [ ] Task 3: **your own** IPO prompt + simple linear regression (`support_info` vs `fairness`)  
- [ ] Saved → Commit & Push → commit message → Commit  
- [ ] Moodle: GitHub link submitted  
